# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to explore a FAIR dataset with multiple record sets using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. We will load metadata and records from a Croissant-compliant dataset, examine its structure, and perform exploratory data analysis (EDA) steps, following a reproducible, standards-based approach.

### Dataset Source
The dataset's metadata and structure are defined by a Croissant schema at the following URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

All references to schema entities—such as record sets and fields—will use their `@id` as per Croissant best practices.

In [ ]:
# Install mlcroissant if not already present
!pip install mlcroissant

## 1. Data Loading

Let's load the dataset metadata and records using `mlcroissant`, referencing the dataset by its Croissant schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata (name, description)
print("Dataset Title:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("Identifier:", dataset.metadata.identifier)
print("Authors:", dataset.metadata.author)
print("Temporal Coverage:", dataset.metadata.temporalCoverage)
print("Spatial Coverage:", dataset.metadata.spatialCoverage)

## 2. Data Overview

Let's enumerate all available record sets, their `@id`s, and the fields (columns) defined for each. This gives us an overview of the dataset's tabular structure.

In [ ]:
# List all record sets and their properties
record_sets = dataset.record_sets  # List of mlc.RecordSet
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"  RecordSet @id: {rs.id}")
    print(f"    Name: {rs.name}")
    print(f"    Description: {getattr(rs, 'description', None)}")
    fields = rs.fields
    print(f"    Fields ({len(fields)}):")
    for field in fields:
        print(f"      - Field @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'dataType', None)}")
    print()

## 3. Data Extraction

Now let's extract records from the main record sets using their `@id`. All loading will be referenced by the `@id` of each RecordSet. We then load these as pandas DataFrames for further analysis.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for RecordSet '@id': {rs_id}, shape: {df.shape}")

# Display available columns from the first record set (if any)
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"\nColumns for RecordSet '@id': {first_rs_id}")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

We will now:
- Filter records based on a numeric field.
- Normalize the numerical values.
- Group by a categorical field, if available.

All selections and manipulations will reference schema objects by their `@id`.

In [ ]:
# Pick the first available record set for EDA (customize as needed)
if record_set_ids:
    rs_id = first_rs_id
    df = dataframes[rs_id]
    print(f"\nExploring RecordSet '@id': {rs_id}")

    # Identify numeric fields by scanning dataTypes
    record_set_obj = next(rs for rs in dataset.record_sets if rs.id == rs_id)
    numeric_field_objs = [f for f in record_set_obj.fields if getattr(f, 'dataType', None) in ['Number', 'Float', 'Integer']]
    if not numeric_field_objs:
        print("No numeric fields found in this record set for EDA.")
    else:
        numeric_field = numeric_field_objs[0]  # Take the first numeric field
        numeric_field_id = numeric_field.id
        print(f"Chosen numeric field for filtering and normalization: '@id': {numeric_field_id}, name: {numeric_field.name}")

        # Make sure field exists and type is appropriate
        if numeric_field_id in df.columns:
            # Convert field to numeric, ignoring errors
            df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
            threshold = df[numeric_field_id].mean()  # Use mean as threshold for demo

            filtered_df = df[df[numeric_field_id] > threshold].copy()
            print(f"Filtered records where {numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)} rows")

            # Normalization
            col_norm = f"{numeric_field_id}_normalized"
            filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Head of normalized numeric field '{col_norm}':")
            display(filtered_df[[numeric_field_id, col_norm]].head())

            # Try to group by the first non-numeric (likely categorical) field
            group_field_objs = [f for f in record_set_obj.fields if getattr(f, 'dataType', None) == 'Text' and f.id in df.columns]
            if group_field_objs:
                group_field = group_field_objs[0].id
                grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame(f"mean_{numeric_field_id}")
                print(f"Grouped means of '{numeric_field_id}' by '{group_field}':")
                display(grouped_df.head())
            else:
                print("No suitable categorical (Text) field found to group by.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and compare group means, if possible.

Note: Please ensure matplotlib is installed if these cells fail to display charts.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field_objs:
    # Distribution plot
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Grouped means bar plot if available
    if 'grouped_df' in locals() and not grouped_df.empty:
        grouped_df_sorted = grouped_df.sort_values(f"mean_{numeric_field_id}", ascending=False).head(10)
        grouped_df_sorted.reset_index(inplace=True)
        plt.figure(figsize=(10,5))
        sns.barplot(data=grouped_df_sorted, x=group_field, y=f"mean_{numeric_field_id}")
        plt.title(f"Top Groups by Mean {numeric_field_id}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we've:
- Loaded Croissant dataset metadata and records via schema URL, referencing all schema entities by `@id`.
- Explored available record sets and their field definitions using `mlcroissant`.
- Extracted records and loaded them into pandas DataFrames.
- Performed basic EDA: filtering, normalization, grouping, and visualization (where suitable fields exist).

This template demonstrates scalable, standards-based data access and exploration using FAIR principles and Croissant schemas. Adapt it by selecting different record sets, fields, or EDA criteria—always referencing entities by their unique `@id` as modeled above.

For more advanced analysis, consult the [mlcroissant documentation](https://github.com/mlcommons/croissant/tree/main/mlcroissant).